# 遥感影像 7 类地物语义分割 · 一键运行 Notebook

**成员 F · AI Studio 部署与现场演示**

流程：环境检查 → 数据挂载/解压 → 训练（已有权重则跳过）→ 测试集评测 → 单图推理出图 → 三模型测速

**使用前提**
1. 左侧「数据集」里已挂载本项目的数据集（只读，路径形如 `/home/aistudio/data/dataXXXXX/`）；
2. 三个模型的权重已放进 `/home/aistudio/work/weights/`（现场演示不训练，直接加载推理）；
3. 若数据集尚未挂载，可先用「上传文件」传少量样图，把 `RUN_TRAIN` 设为 `False` 只跑推理演示。

> 现场演示务必使用「加载权重直接推理」的路径，**不得现场训练**。

In [ ]:
# 1. 环境检查（AI Studio 自带 paddlepaddle-gpu，无需安装；其余依赖见 requirements.txt）
import sys
import paddle

print('Python:', sys.version.split()[0])
print('Paddle:', paddle.__version__)
print('CUDA 编译:', paddle.device.is_compiled_with_cuda())
paddle.utils.run_check()

In [ ]:
# 2. 路径与开关（按实际挂载结果改这里，其余单元格不用动）
import os

WORK = '/home/aistudio/work'
PROJECT = f'{WORK}/Feature-recogniton'          # 项目代码目录
DATA_ZIP = '/home/aistudio/data/dataXXXXX/dataset_locked_v1.zip'  # 改成你的数据集路径；没有就留空
DATA_ROOT = '/home/aistudio/data/dataXXXXX/dataset_locked_v1'     # 已解压好的数据集目录（优先于 zip）
WEIGHTS = f'{WORK}/weights'

RUN_TRAIN = False        # 现场演示保持 False；需要重新训练时改 True
BACKBONE = 'resnet50'    # resnet50 / vgg16_unet / mobilenet_v3
CKPT = f'{WEIGHTS}/{BACKBONE}_best.pth'

TRAIN_IMG = f'{DATA_ROOT}/train/images_png'; TRAIN_MASK = f'{DATA_ROOT}/train/masks_png'
VAL_IMG = f'{DATA_ROOT}/val/images_png';     VAL_MASK = f'{DATA_ROOT}/val/masks_png'
TEST_IMG = f'{DATA_ROOT}/test/images_png';   TEST_MASK = f'{DATA_ROOT}/test/masks_png'

os.makedirs(WEIGHTS, exist_ok=True)
for p in [PROJECT, WEIGHTS]:
    print(p, '存在' if os.path.exists(p) else '不存在')

In [ ]:
# 3. 数据准备：挂载目录只读，先解压到可写的 work/ 下（已解压则跳过）
import os

if not os.path.exists(DATA_ROOT) and DATA_ZIP and os.path.exists(DATA_ZIP):
    !mkdir -p /home/aistudio/work/data
    !unzip -q -o {DATA_ZIP} -d /home/aistudio/work/data
    print('解压完成:', DATA_ROOT)
else:
    print('数据目录状态:', os.path.exists(DATA_ROOT), '| 跳过解压')

if os.path.exists(TRAIN_IMG):
    for split, p in [('train', TRAIN_IMG), ('val', VAL_IMG), ('test', TEST_IMG)]:
        print(f'{split}: {len(os.listdir(p))} 张')

In [ ]:
# 4. 训练（已有权重或 RUN_TRAIN=False 时自动跳过）
import os

if RUN_TRAIN and not os.path.exists(CKPT) and os.path.exists(TRAIN_IMG):
    !cd {PROJECT} && python train.py \
        --backbone {BACKBONE} \
        --train-img {TRAIN_IMG} --train-mask {TRAIN_MASK} \
        --val-img {VAL_IMG} --val-mask {VAL_MASK} \
        --imgsz 512 --epochs 30 --patience 8 --lr 1e-4 \
        --unfreeze-last-stage --augment \
        --outdir {WEIGHTS} --run-name {BACKBONE}_best
else:
    print('跳过训练 | 权重存在:', os.path.exists(CKPT), '| RUN_TRAIN =', RUN_TRAIN)

In [ ]:
# 5. 测试集评测（PA / mIoU / 每类 IoU / P / R）
import os

if os.path.exists(CKPT) and os.path.exists(TEST_IMG):
    !cd {PROJECT} && python eval.py --checkpoint {CKPT} \
        --test-img {TEST_IMG} --test-mask {TEST_MASK}
else:
    print('跳过评测（缺权重或缺测试集）')

In [ ]:
# 6. 单图推理演示（答辩现场用这张出图）
# infer.py 里用了 plt.show()，Notebook 中改为直接内联显示图片
import os
import numpy as np
import paddle
import matplotlib.pyplot as plt
from PIL import Image

from models import build_model
from dataset import build_transform
from metrics import CLASS_NAMES_7

CLASS_COLORS = np.array([
    [245, 245, 245], [220, 20, 60], [128, 128, 128], [30, 144, 255],
    [255, 165, 0], [0, 128, 0], [255, 255, 0],
], dtype=np.uint8)


def demo_infer(image_path, ckpt_path):
    ckpt = paddle.load(ckpt_path, return_numpy=False)
    cfg = ckpt.get('config', {})
    imgsz = int(cfg.get('imgsz', 512))
    model = build_model(cfg.get('backbone', 'resnet50'), cfg.get('n_classes', 7),
                        pretrained=False, freeze_backbone=False,
                        activation=cfg.get('activation', 'relu6'))
    model.set_state_dict(ckpt['model_state_dict'])
    model.eval()

    img = Image.open(image_path).convert('RGB')
    x = build_transform(imgsz)(img).unsqueeze(0)
    with paddle.no_grad():
        pred = paddle.argmax(model(x), axis=1).squeeze(0).cpu().numpy()
    color = CLASS_COLORS[pred]

    fig, axes = plt.subplots(1, 2, figsize=(14, 7))
    axes[0].imshow(img); axes[0].set_title('Original'); axes[0].axis('off')
    axes[1].imshow(color); axes[1].set_title('Prediction'); axes[1].axis('off')
    handles = [plt.Rectangle((0, 0), 1, 1, facecolor=c / 255.0) for c in CLASS_COLORS]
    fig.legend(handles, CLASS_NAMES_7, loc='lower center', ncol=7, fontsize=10)
    plt.tight_layout(); plt.show()

    total = pred.size
    for cls_id, cnt in zip(*np.unique(pred, return_counts=True)):
        print(f'{CLASS_NAMES_7[cls_id]}: {cnt:,} 像素 ({cnt / total * 100:.2f}%)')


# 换成任意一张图即可现场演示
DEMO_IMAGE = f'{TEST_IMG}/' + sorted(os.listdir(TEST_IMG))[0] if os.path.exists(TEST_IMG) else None
if DEMO_IMAGE and os.path.exists(CKPT):
    print('演示图片:', DEMO_IMAGE, '| 权重:', CKPT)
    demo_infer(DEMO_IMAGE, CKPT)
else:
    print('缺演示图片或权重，请检查 DATA_ROOT / WEIGHTS')

In [ ]:
# 7. 三模型测速（参数量 / 文件大小 / 单张耗时），结果写入 benchmark_results.csv
import glob
import os

ckpts = ' '.join(glob.glob(f'{WEIGHTS}/*.pth'))
if ckpts:
    !cd {PROJECT} && python benchmark.py --checkpoints {ckpts} --repeats 50
else:
    !cd {PROJECT} && python benchmark.py --imgsz 512 --repeats 50

---
### 现场演示脚本（照读即可）

1. 运行「1. 环境检查」——证明环境可用（paddle + GPU）；
2. 运行「6. 单图推理演示」——把 `DEMO_IMAGE` 换成老师指定的任意图片，当场出分割结果和类别占比；
3. 打开「7. 三模型测速」的结果表，讲模型大小与推理速度对比；
4. 若被要求换模型演示：改 `BACKBONE` 与 `CKPT` 后重跑单元格 6 即可。